# Inter-Coder Reliability — Step 1

Sentence-level reliability for the binary task of detecting whether a sentence contains any social-group reference. Each outlet's reliability sample was coded independently by two or more annotators (anonymised as A1, A2, A3, ...). Reliability is reported as Krippendorff's α (Krippendorff 2004), computed across all annotators per outlet.

In [1]:
import ast
import json
from itertools import combinations
from pathlib import Path

import pandas as pd
import krippendorff

In [2]:
BASE_DIR = Path('../data/manual_annotations/step_1')
OUTPUT_DIR = Path('./')

OUTLETS = [
    # (display_name, country, folder)
    ('Le Figaro',              'France',  'figaro'),
    ('Le Monde',               'France',  'monde'),
    ('Le Monde diplomatique',  'France',  'mondediplo'),
    ('Le Parisien',            'France',  'parisien'),
    ('Médiapart',              'France',  'mediapart'),
    ('Libération',             'France',  'libe'),
    ('Bild',                   'Germany', 'bild'),
    ('Frankfurter Allgemeine', 'Germany', 'faz'),
    ('Spiegel',                'Germany', 'spiegel'),
    ('Süddeutsche Zeitung',    'Germany', 'sz'),
    ('Die Welt',               'Germany', 'welt'),
    ('Die Zeit',               'Germany', 'zeit'),
]

## Data loading

Each outlet folder contains one Doccano JSONL file per annotator. Files within an outlet are row-aligned by sentence position. For each sentence, the annotator's spans are reduced to a binary indicator (1 if any span was marked, 0 otherwise).

In [3]:
EMPTY_TOKENS = {0, 0.0, '0', '', '[]', '0.0', 'nan', None}


def has_annotation(label_field) -> int:
    if isinstance(label_field, list):
        return 1 if any(isinstance(s, list) and len(s) >= 2 for s in label_field) else 0
    try:
        if pd.isna(label_field):
            return 0
    except (TypeError, ValueError):
        pass
    if label_field in EMPTY_TOKENS:
        return 0
    s = str(label_field).strip()
    if s in EMPTY_TOKENS:
        return 0
    try:
        parsed = ast.literal_eval(s)
    except (ValueError, SyntaxError):
        return 0
    return 1 if isinstance(parsed, list) and any(
        isinstance(x, list) and len(x) >= 2 for x in parsed
    ) else 0


def load_outlet(folder_name: str):
    """Load all annotator JSONL files from an outlet folder.

    Returns (n_annotators, binary_matrix). Annotator identities are not retained;
    files are loaded in alphabetical order and treated as anonymous rows.
    """
    folder = BASE_DIR / folder_name
    files = sorted(folder.glob('*.jsonl'))
    if not files:
        raise FileNotFoundError(f'No JSONL files in {folder}')

    rows_by_file = []
    for p in files:
        with open(p, 'r', encoding='utf-8') as f:
            rows_by_file.append([json.loads(line) for line in f if line.strip()])

    n = min(len(rs) for rs in rows_by_file)
    binary_matrix = [[has_annotation(rs[i].get('label', [])) for i in range(n)]
                     for rs in rows_by_file]
    return len(files), binary_matrix

## Per-outlet α calculation

In [4]:
results = []
loaded = {}

for display, country, folder in OUTLETS:
    n_annotators, binary_matrix = load_outlet(folder)
    alpha = round(
        float(krippendorff.alpha(reliability_data=binary_matrix, level_of_measurement='nominal')),
        4,
    )
    results.append({
        'Outlet':         display,
        'Country':        country,
        'Krippendorff α': alpha,
        '# Sentences':    len(binary_matrix[0]),
        '# Annotators':   n_annotators,
    })
    loaded[display] = (n_annotators, binary_matrix)

df_outlets = pd.DataFrame(results)
df_outlets.to_csv(OUTPUT_DIR / 'icr_step1_by_outlet.csv', index=False)
df_outlets

,Outlet,Country,Krippendorff α,# Sentences,# Annotators
0,Le Figaro,France,0.9141,300,2
1,Le Monde,France,0.9444,300,2
2,Le Monde diplomatique,France,0.8944,1200,2
3,Le Parisien,France,0.6541,300,2
4,Médiapart,France,0.7442,1200,2
5,Libération,France,0.7568,1206,4
6,Bild,Germany,0.6757,300,5
7,Frankfurter Allgemeine,Germany,0.7035,300,2
8,Spiegel,Germany,0.6820,300,2
9,Süddeutsche Zeitung,Germany,0.7431,300,2


## Printout

In [5]:
print(f'{"Newspaper":<28} {"α":>6}  {"N":>6}  {"coders":>6}')
print('-' * 56)
for country_name, code in [('French Newspapers', 'France'), ('German Newspapers', 'Germany')]:
    print(country_name)
    sub = df_outlets[df_outlets['Country'] == code]
    for _, r in sub.iterrows():
        print(f'  {r["Outlet"]:<26} {r["Krippendorff α"]:>6.2f}  '
              f'{r["# Sentences"]:>6,}  {r["# Annotators"]:>6}')

Newspaper                         α       N  coders
--------------------------------------------------------
French Newspapers
  Le Figaro                    0.91     300       2
  Le Monde                     0.94     300       2
  Le Monde diplomatique        0.89   1,200       2
  Le Parisien                  0.65     300       2
  Médiapart                    0.74   1,200       2
  Libération                   0.76   1,206       4
German Newspapers
  Bild                         0.68     300       5
  Frankfurter Allgemeine       0.70     300       2
  Spiegel                      0.68     300       2
  Süddeutsche Zeitung          0.74     300       2
  Die Welt                     0.63     300       2
  Die Zeit                     0.75     300       5


## Pairwise breakdown (outlets with more than two annotators)

In [6]:
pair_rows = []
for display, (n_ann, matrix) in loaded.items():
    if n_ann < 3:
        continue
    country = next(c for d, c, _ in OUTLETS if d == display)
    for i, j in combinations(range(n_ann), 2):
        pair_alpha = round(
            float(krippendorff.alpha(
                reliability_data=[matrix[i], matrix[j]],
                level_of_measurement='nominal',
            )),
            4,
        )
        pair_rows.append({
            'Outlet':         display,
            'Country':        country,
            'Pair':           f'A{i+1} + A{j+1}',
            'Krippendorff α': pair_alpha,
            'N':              len(matrix[0]),
        })

df_pairs = pd.DataFrame(pair_rows)
df_pairs.to_csv(OUTPUT_DIR / 'icr_step1_by_annotator_pair.csv', index=False)
df_pairs.set_index(['Outlet', 'Pair'])

Country  Krippendorff α     N
Outlet     Pair                                  
Libération A1 + A2   France          0.7526  1206
           A1 + A3   France          0.7221  1206
           A1 + A4   France          0.8469  1206
           A2 + A3   France          0.7225  1206
           A2 + A4   France          0.7643  1206
           A3 + A4   France          0.7248  1206
Bild       A1 + A2  Germany          0.7506   300
           A1 + A3  Germany          0.7152   300
           A1 + A4  Germany          0.5919   300
           A1 + A5  Germany          0.6003   300
           A2 + A3  Germany          0.9051   300
           A2 + A4  Germany          0.5709   300
           A2 + A5  Germany          0.7368   300
           A3 + A4  Germany          0.5556   300
           A3 + A5  Germany          0.7203   300
           A4 + A5  Germany          0.5732   300
Die Zeit   A1 + A2  Germany          0.7561   300
           A1 + A3  Germany          0.8051   300
           A1 + A4  Germany          0.7524   300
           A1 + A5  Germany          0.6912   300
           A2 + A3  Germany          0.8128   300
           A2 + A4  Germany          0.8260   300
           A2 + A5  Germany          0.7330   300
           A3 + A4  Germany          0.7580   300
           A3 + A5  Germany          0.7134   300
           A4 + A5  Germany          0.6961   300